## Video Inference

Given entire pipeline, we now want to expand our model deployment to also support video. This looks like:

- Input: Video file (.mp4)
- Output: Video file overlayed with bounding box and keypoints for each individual frame. Prediction of each frame's current pose under bounding box. Exported as (.mp4) file

## Implementation

### 1) Setup

In [7]:
from pathlib import Path
import sys
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from models.bbox_detection import BBoxDetectionModel
from models.keypoint_detection import KeypointDetectionModel
from models.pose_classification import PoseClassificationModel
from preprocessing.pil_preprocessing import crop_pil, letterbox_resize, norm_bbox_to_xyxy_pixels
from preprocessing.tensor_preprocessing import heatmaps_to_keypoints, normalize_keypoints_xy

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "inference":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Repo root:", PROJECT_ROOT)
print("Device:", DEVICE)

Repo root: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection
Device: cpu


In [ ]:
INPUT_VIDEO_PATH = PROJECT_ROOT / "datasets" / "orvile" / "sample.mp4"
OUTPUT_VIDEO_PATH = PROJECT_ROOT / "evaluation_outputs" / f"{INPUT_VIDEO_PATH.stem}_predictions.mp4"

BBOX_CKPT_PATH = PROJECT_ROOT / "exports" / "bbox_best.pt"
KEYPOINT_CKPT_PATH = PROJECT_ROOT / "exports" / "keypoint_best_state_dict.pt"
POSE_CKPT_PATH = PROJECT_ROOT / "exports" / "pose_best.pt"

BBOX_IMAGE_SIZE = (256, 256)

OUTPUT_VIDEO_PATH.parent.mkdir(parents=True, exist_ok=True)
print("Input:", INPUT_VIDEO_PATH)
print("Output:", OUTPUT_VIDEO_PATH)

Input: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/datasets/orvile/sample.mp4
Output: /Users/hanyiliu/Documents/GitHub/tennis-pose-detection/evaluation_outputs/sample_predictions.mp4


### 2) Load Models

In [9]:
bbox_model = BBoxDetectionModel().to(DEVICE)
bbox_state = torch.load(BBOX_CKPT_PATH, map_location=DEVICE)
if isinstance(bbox_state, dict) and "model_state" in bbox_state:
    bbox_model.load_state_dict(bbox_state["model_state"])
else:
    bbox_model.load_state_dict(bbox_state)
bbox_model.eval()

keypoint_payload = torch.load(KEYPOINT_CKPT_PATH, map_location=DEVICE)
keypoint_num_keypoints = int(keypoint_payload["num_keypoints"])
kp_h = int(keypoint_payload.get("image_height"))
kp_w = int(keypoint_payload.get("image_width"))
KEYPOINT_IMAGE_SIZE = (kp_h, kp_w)
keypoint_model = KeypointDetectionModel(num_keypoints=keypoint_num_keypoints).to(DEVICE)
keypoint_model.load_state_dict(keypoint_payload["model_state"])
keypoint_model.eval()

pose_payload = torch.load(POSE_CKPT_PATH, map_location=DEVICE)
pose_label_names = pose_payload.get("label_names", ["backhand", "forehand", "ready_position", "serve"])
pose_model = PoseClassificationModel(num_keypoints=keypoint_num_keypoints, num_classes=len(pose_label_names)).to(DEVICE)
if isinstance(pose_payload, dict) and "model_state" in pose_payload:
    pose_model.load_state_dict(pose_payload["model_state"])
else:
    pose_model.load_state_dict(pose_payload)
pose_model.eval()

print("Loaded bbox, keypoint, and pose models.")
print("Keypoint image size:", KEYPOINT_IMAGE_SIZE)
print("Pose labels:", pose_label_names)

Loaded bbox, keypoint, and pose models.
Keypoint image size: (128, 128)
Pose labels: ['Backhand', 'Forehand', 'Ready_Position', 'Serve']


### 3) Geometry + Frame Inference Helpers

In [10]:
def letterbox_params(orig_w: int, orig_h: int, target_h: int, target_w: int):
    scale = min(target_w / float(orig_w), target_h / float(orig_h))
    new_w = max(1, int(round(orig_w * scale)))
    new_h = max(1, int(round(orig_h * scale)))
    pad_x = (target_w - new_w) / 2.0
    pad_y = (target_h - new_h) / 2.0
    return scale, pad_x, pad_y


def to_pil_rgb(frame_bgr: np.ndarray) -> Image.Image:
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(frame_rgb)


def safe_bbox_xyxy(x1: float, y1: float, x2: float, y2: float, width: int, height: int):
    x1 = int(np.clip(round(x1), 0, width - 1))
    y1 = int(np.clip(round(y1), 0, height - 1))
    x2 = int(np.clip(round(x2), 0, width - 1))
    y2 = int(np.clip(round(y2), 0, height - 1))
    if x2 <= x1:
        x2 = min(width - 1, x1 + 1)
    if y2 <= y1:
        y2 = min(height - 1, y1 + 1)
    return x1, y1, x2, y2


def infer_single_frame(frame_bgr: np.ndarray):
    frame_h, frame_w = frame_bgr.shape[:2]
    pil_frame = to_pil_rgb(frame_bgr)

    bbox_in = letterbox_resize(pil_frame, BBOX_IMAGE_SIZE)
    bbox_tensor = torch.from_numpy(np.array(bbox_in).transpose(2, 0, 1)).float().div(255.0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        bbox_pred = bbox_model(bbox_tensor)[0].detach().cpu().numpy()

    x1, y1, x2, y2 = norm_bbox_to_xyxy_pixels(bbox_pred, frame_w, frame_h)
    x1, y1, x2, y2 = safe_bbox_xyxy(x1, y1, x2, y2, frame_w, frame_h)

    crop_w = max(1, x2 - x1)
    crop_h = max(1, y2 - y1)
    crop = crop_pil(pil_frame, (x1, y1, x2, y2))
    kp_in = letterbox_resize(crop, KEYPOINT_IMAGE_SIZE)
    kp_tensor = torch.from_numpy(np.array(kp_in).transpose(2, 0, 1)).float().div(255.0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        heatmaps = keypoint_model(kp_tensor)

    kp_xyc = heatmaps_to_keypoints(heatmaps, image_size=KEYPOINT_IMAGE_SIZE)[0].detach().cpu().numpy()

    norm_xy = normalize_keypoints_xy(torch.from_numpy(kp_xyc).unsqueeze(0), image_size=KEYPOINT_IMAGE_SIZE)[0].numpy()
    pose_features = np.concatenate([norm_xy, kp_xyc[:, 2:3]], axis=1)
    pose_tensor = torch.from_numpy(pose_features).float().unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pose_logits = pose_model(pose_tensor)
        pose_probs = torch.softmax(pose_logits, dim=-1)[0].detach().cpu().numpy()

    pose_idx = int(np.argmax(pose_probs))
    pose_label = pose_label_names[pose_idx]
    pose_conf = float(pose_probs[pose_idx])

    kp_h, kp_w = KEYPOINT_IMAGE_SIZE
    scale, pad_x, pad_y = letterbox_params(crop_w, crop_h, kp_h, kp_w)

    keypoints_frame = []
    for x_kp, y_kp, c_kp in kp_xyc:
        x_crop = (float(x_kp) - pad_x) / scale
        y_crop = (float(y_kp) - pad_y) / scale
        x_frame = np.clip(x1 + x_crop, 0, frame_w - 1)
        y_frame = np.clip(y1 + y_crop, 0, frame_h - 1)
        keypoints_frame.append((float(x_frame), float(y_frame), float(c_kp)))

    return {
        "bbox": (x1, y1, x2, y2),
        "keypoints": keypoints_frame,
        "pose_label": pose_label,
        "pose_conf": pose_conf,
    }


def draw_prediction(frame_bgr: np.ndarray, pred: dict, kp_conf_threshold: float = 0.1) -> np.ndarray:
    annotated = frame_bgr.copy()
    x1, y1, x2, y2 = pred["bbox"]

    cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)

    text = f"{pred['pose_label']} ({pred['pose_conf']:.2f})"
    text_y = y2 + 22
    if text_y >= annotated.shape[0]:
        text_y = max(18, y1 - 8)
    cv2.putText(annotated, text, (x1, text_y), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2, cv2.LINE_AA)

    for x_kp, y_kp, c_kp in pred["keypoints"]:
        if c_kp >= kp_conf_threshold:
            cv2.circle(annotated, (int(round(x_kp)), int(round(y_kp))), 3, (0, 0, 255), -1)

    return annotated

### 4) Run Video Inference + Export

In [ ]:
if not INPUT_VIDEO_PATH.exists():
    raise FileNotFoundError(f"Input video not found: {INPUT_VIDEO_PATH}")

cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Could not open input video: {INPUT_VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 0:
    fps = 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(OUTPUT_VIDEO_PATH), fourcc, fps, (width, height))
if not writer.isOpened():
    cap.release()
    raise RuntimeError(f"Could not open output writer: {OUTPUT_VIDEO_PATH}")

processed = 0
start_time = time.time()

while True:
    ok, frame = cap.read()
    if not ok:
        break

    pred = infer_single_frame(frame)
    annotated = draw_prediction(frame, pred)
    writer.write(annotated)
    processed += 1

cap.release()
writer.release()

elapsed = max(1e-6, time.time() - start_time)
print(f"Processed frames: {processed}/{frame_count}")
print(f"Elapsed: {elapsed:.2f}s | Throughput: {processed / elapsed:.2f} fps")
print(f"Saved: {OUTPUT_VIDEO_PATH}")

### 5) Validate Output

In [ ]:
in_cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))
out_cap = cv2.VideoCapture(str(OUTPUT_VIDEO_PATH))

in_meta = {
    "fps": in_cap.get(cv2.CAP_PROP_FPS),
    "width": int(in_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "height": int(in_cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "frames": int(in_cap.get(cv2.CAP_PROP_FRAME_COUNT)),
}
out_meta = {
    "fps": out_cap.get(cv2.CAP_PROP_FPS),
    "width": int(out_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "height": int(out_cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "frames": int(out_cap.get(cv2.CAP_PROP_FRAME_COUNT)),
}

in_cap.release()
out_cap.release()

print("Input metadata:", in_meta)
print("Output metadata:", out_meta)

In [ ]:
cap = cv2.VideoCapture(str(OUTPUT_VIDEO_PATH))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
indices = sorted(set([0, max(0, total // 2), max(0, total - 1)]))

frames = []
for idx in indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if ok:
        frames.append((idx, cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
cap.release()

if frames:
    plt.figure(figsize=(16, 5))
    for i, (idx, frame_rgb) in enumerate(frames, start=1):
        plt.subplot(1, len(frames), i)
        plt.imshow(frame_rgb)
        plt.title(f"Frame {idx}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No frames available for preview.")

### Usage

- Set `INPUT_VIDEO_PATH` and `OUTPUT_VIDEO_PATH` in Setup.
- Run all cells from top to bottom.
- Open the generated output mp4 in `evaluation_outputs/`.